# Named Entity Recognition (NER) on Financial News: Recurrent Neural Network Comparison (PyTorch + CUDA)

In financial news, sentences are long, complex, and full of domain-specific terminology and ambiguous words. For example, the word **'Apple'** could refer to a fruit, or it could refer to a tech company. Context is everything. This project compares 7 Recurrent Neural Network (RNN) architectures implemented in **PyTorch** and accelerated using **CUDA** on an **NVIDIA GeForce RTX 3050 Laptop GPU** to perform NER on financial news texts:

1. **SimpleRNN (Baseline)**: Single vanilla recurrent layer. Shows the impact of vanishing gradients.
2. **LSTM**: Single Long Short-Term Memory layer. Utilizes gates to maintain long-range dependencies.
3. **GRU**: Single Gated Recurrent Unit layer. A streamlined version of LSTM with fewer parameters.
4. **Bi-directional RNN**: Wraps LSTM in a bidirectional wrapper to capture both past and future context.
5. **Bidirectional GRU**: Wraps GRU in a bidirectional wrapper for efficient, bidirectional context tracking.
6. **Attention LSTM**: Integrates a sequence self-attention layer on top of LSTM outputs to selectively focus on key context words.
7. **Deep RNN**: Two stacked LSTM layers to learn hierarchical representations.

### Project Metrics
We systematically track and compare the following metrics:
- **Entity-Level F1-Score (seqeval)**: Pure performance on detecting named entities (PER, LOC, ORG) excluding padding.
- **Training Time per Epoch**: Computational cost of training.
- **Inference Latency**: Milliseconds taken to predict tags for a single sentence (critical for APIs, CUDA-synchronized).
- **Learning Curves**: Convergence rate and overfitting behavior.

## 1. Imports and Environment Setup

We import all necessary packages, check CUDA accessibility, and import our custom source scripts.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import torch
import torch.nn as nn

print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

# Import local modules
from src.data_preprocessing import load_and_preprocess_data
from src.models import SimpleRNNModel, LSTMModel, GRUModel, BidirectionalModel, DeepRNNModel, BidirectionalGRUModel, AttentionLSTMModel
from src.evaluate import evaluate_model

print('All libraries and custom modules imported successfully!')

## 2. Dataset Ingestion and Exploration

We use the **`gtfintechlab/finer-ord-bio`** dataset from Hugging Face. This dataset contains tokenized financial news sentences annotated with BIO tags for three categories: **Person (PER)**, **Location (LOC)**, and **Organization (ORG)**.

In [ ]:
# Load and preprocess data using our preprocessing module
(x_train, y_train), (x_val, y_val), (x_test, y_test), metadata = load_and_preprocess_data(
    max_len=128, vocab_size=15000
)

vocab_size = metadata['vocab_size']
num_classes = metadata['num_classes']
label_list = metadata['label_list']

print(f'Train sentences: {len(x_train)}')
print(f'Val sentences: {len(x_val)}')
print(f'Test sentences: {len(x_test)}')
print(f'NER tags list: {label_list}')

### Sample Data Inspection

Let's see what a padded sentence and its corresponding label sequence look like (tags padded with -1).

In [ ]:
idx = 10
vocab_rev = {idx: word for word, idx in metadata['vocab'].items()}
words = [vocab_rev.get(w_idx, '<UNK>') for w_idx in x_train[idx] if w_idx != 0]
tags = [label_list[t_idx] for t_idx in y_train[idx] if t_idx != -1]

print('Tokens:', words)
print('Labels:', tags)

## 3. Models Definition

We load the builder functions defined in `src/models.py`. To ensure that padded sequences do not impact learning, all models utilize PyTorch's `nn.Embedding(..., padding_idx=0)`.

In [ ]:
simple_rnn = SimpleRNNModel(vocab_size, num_classes)
lstm = LSTMModel(vocab_size, num_classes)
gru = GRUModel(vocab_size, num_classes)
bidirectional = BidirectionalModel(vocab_size, num_classes)
bidirectional_gru = BidirectionalGRUModel(vocab_size, num_classes)
attention_lstm = AttentionLSTMModel(vocab_size, num_classes)
deep_rnn = DeepRNNModel(vocab_size, num_classes)

print('Model architectures successfully defined!')

## 4. Benchmark Performance and Analysis

We trained the five models for 10 epochs on the entire training split and saved their histories and comparison scores. Let's load the comparison metrics table.

In [ ]:
df_metrics = pd.read_csv(os.path.join('models', 'comparison_metrics.csv'))
ipd.display(df_metrics)

## 5. Visualizing Comparison Results

Let's visualize the loss convergence over training epochs.

In [ ]:
print('Displaying Learning Curves (Loss & Accuracy):')
ipd.display(ipd.Image(filename=os.path.join('models', 'learning_curves.png')))

### Accuracy vs Latency Trade-off

Let's visualize the entity-level F1-score compared against inference latency.

In [ ]:
print('Displaying Accuracy and Latency Comparison:')
ipd.display(ipd.Image(filename=os.path.join('models', 'tradeoff_comparison.png')))

## 6. Analysis & Discussion

Based on our results, here are key insights for each architecture:

1. **SimpleRNN**: Suffers heavily from vanishing gradients. While training and inference are very fast, it struggles to remember the contexts of long financial sentences, leading to the lowest entity F1-Score.
2. **LSTM**: Gating structures show a massive jump in F1-score compared to SimpleRNN. It successfully tracks relationships from the beginning to the end of a sentence.
3. **GRU**: Achieves similar performance to the LSTM model but trains significantly faster. With fewer gate operations, it requires less computation.
4. **Bidirectional RNN**: Wrapping the LSTM in a Bidirectional wrapper gives a substantial boost in accuracy. Because future context defines named entities (e.g. knowing what comes after 'Amazon' to distinguish locations from companies), reading forwards and backwards simultaneously is extremely powerful.
5. **Deep RNN**: Stacking two LSTM layers captures hierarchical semantics, yielding the absolute highest F1-Score. However, it also comes with the highest computational complexity, leading to increased training and inference latencies.